In [ ]:
import numpy as np
import pandas as pd
import scipy.io as scio
from pathlib import Path

# ====== 改成你的 MI1 路径 ======
mi1_dir = Path("../data/MI1")   # 例如 Path("./MI1")

mat_files = sorted(mi1_dir.glob("*.mat"))
if len(mat_files) == 0:
    raise ValueError(f"No .mat files found in {mi1_dir}")
70~73
summary_rows = []

for mat_path in mat_files:
    data = scio.loadmat(mat_path)

    # ---- 基本字段检查 ----
    missing_keys = [k for k in ["X", "y", "session"] if k not in data]
    if missing_keys:
        summary_rows.append({
            "file": mat_path.name,
            "status": f"missing keys: {missing_keys}",
        })
        continue

    X = data["X"]
    y_task = np.asarray(data["y"]).reshape(-1)
    y_session_raw = np.asarray(data["session"]).reshape(-1)

    # session 常常是 object / str，这里统一转成可读字符串，再尽量转 int
    y_session_str = [str(v).strip() for v in y_session_raw]
    try:
        y_session = np.array([int(v) for v in y_session_str], dtype=np.int64)
        session_display = y_session
    except Exception:
        session_display = np.array(y_session_str, dtype=object)

    # ---- 长度一致性检查 ----
    n_x = X.shape[0]
    n_y = len(y_task)
    n_s = len(y_session_raw)
    aligned = (n_x == n_y == n_s)

    # ---- task 分布 ----
    uniq_task, cnt_task = np.unique(y_task, return_counts=True)
    task_dist = ", ".join([f"{int(k)}:{int(v)}" for k, v in zip(uniq_task, cnt_task)])

    # ---- session 分布 ----
    uniq_session, cnt_session = np.unique(session_display, return_counts=True)
    session_dist = ", ".join([f"{k}:{int(v)}" for k, v in zip(uniq_session, cnt_session)])

    # ---- task × session 交叉分布 ----
    # 看每个 session 下不同 task 的 trial 数，便于判断对应关系是否异常
    cross_df = pd.crosstab(
        pd.Series(session_display, name="session"),
        pd.Series(y_task, name="task"),
        dropna=False
    )

    # 转成紧凑字符串
    cross_parts = []
    for sess in cross_df.index:
        row_parts = [f"task{col}={int(cross_df.loc[sess, col])}" for col in cross_df.columns]
        cross_parts.append(f"session {sess}: " + ", ".join(row_parts))
    task_session_dist = " | ".join(cross_parts)

    # ---- 补充形状信息 ----
    x_shape = tuple(X.shape)
    x_dtype = str(X.dtype)

    # trial 间差异的粗统计：看每个 trial 的均值/标准差范围
    # 不是逐 trial 列出来，而是做整体概括
    if X.ndim == 3:
        trial_mean = X.mean(axis=(1, 2))
        trial_std = X.std(axis=(1, 2))
        mean_range = f"[{trial_mean.min():.4f}, {trial_mean.max():.4f}]"
        std_range = f"[{trial_std.min():.4f}, {trial_std.max():.4f}]"
        mean_std = f"{trial_mean.mean():.4f}"
        std_std = f"{trial_std.mean():.4f}"
    else:
        mean_range = "N/A"
        std_range = "N/A"
        mean_std = "N/A"
        std_std = "N/A"

    summary_rows.append({
        "file": mat_path.name,
        "status": "ok" if aligned else "length mismatch",
        "X_shape": x_shape,
        "X_dtype": x_dtype,
        "n_trials_X": n_x,
        "n_y_task": n_y,
        "n_session": n_s,
        "aligned_X_y_session": aligned,
        "task_labels": list(map(int, uniq_task.tolist())),
        "task_dist": task_dist,
        "session_labels": list(map(str, uniq_session.tolist())),
        "session_dist": session_dist,
        "task_x_session_dist": task_session_dist,
        "trial_mean_range": mean_range,
        "trial_std_range": std_range,
        "avg_trial_mean": mean_std,
        "avg_trial_std": std_std,
    })

summary_df = pd.DataFrame(summary_rows)

# ====== 主汇总表 ======
pd.set_option("display.max_colwidth", 200)
display(summary_df)

# ====== 额外：只看有问题的文件 ======
problem_df = summary_df[summary_df["aligned_X_y_session"] != True] if "aligned_X_y_session" in summary_df.columns else pd.DataFrame()
if len(problem_df) > 0:
    print("发现长度不一致的文件：")
    display(problem_df)
else:
    print("所有 mat 文件的 X / y / session 长度都一致。")

# ====== 全局汇总（跨所有 mat） ======
all_task_counter = {}
all_session_counter = {}
global_rows = []

for mat_path in mat_files:
    data = scio.loadmat(mat_path)
    if not all(k in data for k in ["X", "y", "session"]):
        continue

    y_task = np.asarray(data["y"]).reshape(-1)
    y_session_raw = np.asarray(data["session"]).reshape(-1)
    y_session_str = [str(v).strip() for v in y_session_raw]

    try:
        y_session = np.array([int(v) for v in y_session_str], dtype=np.int64)
    except Exception:
        y_session = np.array(y_session_str, dtype=object)

    for k, v in zip(*np.unique(y_task, return_counts=True)):
        all_task_counter[int(k)] = all_task_counter.get(int(k), 0) + int(v)

    for k, v in zip(*np.unique(y_session, return_counts=True)):
        all_session_counter[str(k)] = all_session_counter.get(str(k), 0) + int(v)

    tmp = pd.DataFrame({
        "task": y_task.astype(int),
        "session": y_session.astype(str),
    })
    global_rows.append(tmp)

print("\n=== 全局 task 分布 ===")
display(pd.DataFrame(
    [{"task": k, "count": v} for k, v in sorted(all_task_counter.items())]
))

print("=== 全局 session 分布 ===")
display(pd.DataFrame(
    [{"session": k, "count": v} for k, v in sorted(all_session_counter.items(), key=lambda x: x[0])]
))

if len(global_rows) > 0:
    global_df = pd.concat(global_rows, ignore_index=True)
    print("=== 全局 task × session 交叉表 ===")
    display(pd.crosstab(global_df["session"], global_df["task"]))

,file,status,X_shape,X_dtype,n_trials_X,n_y_task,n_session,aligned_X_y_session,task_labels,task_dist,session_labels,session_dist,task_x_session_dist,trial_mean_range,trial_std_range,avg_trial_mean,avg_trial_std
0,1.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:23, 1:22","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=8, task1=7 | session 2: task0=8, task1=7 | session 3: task0=7, task1=8","[-0.0000, 0.0000]","[0.0000, 0.0000]",0.0000,0.0000
1,10.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:24, 1:21","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=8, task1=7 | session 2: task0=8, task1=7 | session 3: task0=8, task1=7","[-0.0000, 0.0000]","[0.0000, 0.0000]",0.0000,0.0000
2,101.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:22, 1:23","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=8, task1=7 | session 2: task0=7, task1=8 | session 3: task0=7, task1=8","[-0.0000, 0.0000]","[0.0000, 0.0000]",0.0000,0.0000
3,102.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:22, 1:23","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=7, task1=8 | session 2: task0=7, task1=8 | session 3: task0=8, task1=7","[-0.0000, 0.0000]","[0.0000, 0.0000]",0.0000,0.0000
4,103.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:22, 1:23","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=7, task1=8 | session 2: task0=8, task1=7 | session 3: task0=7, task1=8","[-0.0000, 0.0000]","[0.0000, 0.0001]",0.0000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,95.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:23, 1:22","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=7, task1=8 | session 2: task0=8, task1=7 | session 3: task0=8, task1=7","[-0.0000, 0.0000]","[0.0000, 0.0001]",-0.0000,0.0000
101,96.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:23, 1:22","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=7, task1=8 | session 2: task0=8, task1=7 | session 3: task0=8, task1=7","[-0.0000, 0.0000]","[0.0000, 0.0001]",-0.0000,0.0000
102,97.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:23, 1:22","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=8, task1=7 | session 2: task0=7, task1=8 | session 3: task0=8, task1=7","[-0.0000, 0.0000]","[0.0000, 0.0001]",-0.0000,0.0000
103,98.mat,ok,"(45, 64, 257)",float32,45,45,45,True,"[0, 1]","0:23, 1:22","[1, 2, 3]","1:15, 2:15, 3:15","session 1: task0=8, task1=7 | session 2: task0=8, task1=7 | session 3: task0=7, task1=8","[-0.0000, 0.0000]","[0.0000, 0.0000]",-0.0000,0.0000


所有 mat 文件的 X / y / session 长度都一致。

=== 全局 task 分布 ===


,task,count
0,0,2384
1,1,2341


=== 全局 session 分布 ===


,session,count
0,1,1575
1,2,1575
2,3,1575


=== 全局 task × session 交叉表 ===


task,0,1
session,,
1,798,777
2,793,782
3,793,782
